In [ ]:
import sys, os, subprocess

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)

try:
    import pyspark
    print("PySpark already installed:", pyspark.__version__)
except ImportError:
    print("PySpark NOT installed — install with: !pip install -q pyspark==4.0.4  (then restart kernel)")

print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

**Step 2: Set environment variables**



In [ ]:
import os, sys

os.environ["JAVA_HOME"]           = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_PYTHON"]      = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("JAVA_HOME         :", os.environ["JAVA_HOME"])
print("PYSPARK_PYTHON    :", os.environ["PYSPARK_PYTHON"])

**Step 3: Create the SparkSession**

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("cipherstop-ingestion")
    .config("spark.driver.memory", "20g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.files.openCostInBytes", str(4 * 1024 * 1024))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark (Python API):", spark.version)
print("Spark (JVM)       :", spark.sparkContext._jsc.sc().version())
print("Master            :", spark.sparkContext.master)

In [ ]:
import shutil

s = spark.range(1_000_000).selectExpr("sum(id) as s").first()["s"]
assert s == 499_999_500_000, f"got {s}"
print("arithmetic OK:", s)

spark.range(1000).write.mode("overwrite").parquet("/kaggle/working/_smoke")
n = spark.read.parquet("/kaggle/working/_smoke").count()
assert n == 1000, f"got {n}"
shutil.rmtree("/kaggle/working/_smoke")
print("parquet I/O OK")
print(" Spark is operational")

In [ ]:
RAW_ROOT = ("/kaggle/input/datasets/hiranomanabu/"
            "ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset")

STREAMS      = ["ata_read", "ata_write", "mem_exec",
                "mem_readwrite", "mem_read", "mem_write"]
HAS_ENTROPY  = {"ata_write", "mem_readwrite", "mem_write"}
EVENT_DDL    = "sec long, ns double, addr long, size long, entropy double, c6 double"
WINDOW_S     = 1.0

RANSOM_FAMILIES = {"Conti", "LockBit", "Darkside", "Ryuk", "REvil", "WannaCry"}

# Hardware holdout (edit to match your final split decision)
VAL_HW = {
    "celeron-gen6/ddr4-1600-16g",
    "i3-gen12/ddr4-2133-16g",
}
TEST_HW = {
    "celeron-gen6/ddr4-2133-8g",
    "i3-gen12/ddr4-3200-16g",
    "i3-gen12/ddr4-2666-8g",
}
ZERODAY_FAMILY = "Ryuk"     # adjust if your zero-day family is different

print("RAW_ROOT exists:", __import__("os").path.isdir(RAW_ROOT))

In [ ]:
import os, re, time, pandas as pd

NAME_RE = re.compile(r"^(.+)-(\d{8}_\d{2}-\d{2}-\d{2})$")

def find_execution_dirs(base):
    """Every directory containing at least one recognised stream CSV."""
    hits = []
    for root, _, files in os.walk(base):
        csvs = {f[:-4] for f in files if f.endswith(".csv")}
        if csvs & set(STREAMS):
            hits.append((root, csvs))
    return hits

t0 = time.time()
exec_dirs = find_execution_dirs(RAW_ROOT)
print(f"scanned in {time.time()-t0:.1f}s | {len(exec_dirs)} execution folders")

rows = []
for root, csvs in exec_dirs:
    rel_full = os.path.relpath(root, RAW_ROOT).replace(os.sep, "/")   # starts with original/extra/...
    parts = rel_full.split("/")
    # Expected depth: top / cpu / ram / activity / execution  (5 parts)
    top, cpu, ram, activity, execution = parts[0], parts[1], parts[2], parts[3], parts[4]

    mt = NAME_RE.match(execution)
    sample = mt.group(1) if mt else execution
    family = re.sub(r"[_-]?\d+$", "", sample)

    sizes = {s: 0 for s in STREAMS}
    for f in os.listdir(root):
        if f.endswith(".csv") and f[:-4] in STREAMS:
            sizes[f[:-4]] = os.path.getsize(os.path.join(root, f))

    rows.append({
        "rel_dir":   rel_full,
        "top":       top, "cpu": cpu, "ram": ram,
        "hw":        f"{cpu}/{ram}",
        "activity":  activity, "execution": execution,
        "sample":    sample, "family": family,
        "has_ata":   int(sizes["ata_read"] > 0 and sizes["ata_write"] > 0),
        "total_bytes": sum(sizes.values()),
    })

m = pd.DataFrame(rows)
m.to_csv("/kaggle/working/manifest_raw.csv", index=False)

print(f"executions: {len(m)} | total size: {m.total_bytes.sum()/1e9:.2f} GB")
print("\ntop:")
print(m.top.value_counts().to_string())
print("\nhw:")
print(m.hw.value_counts().to_string())
print("\nfamilies:")
print(m.family.value_counts().to_string())

In [ ]:
import pandas as pd

m = pd.read_csv("/kaggle/working/manifest_raw.csv")

def split_of(row):
    # zero-day family first (safety)
    if row.family == ZERODAY_FAMILY:            return "zeroday"
    if row.top == "mix":                        return "mix"       # eval only
    if row.top == "variants":                   return "variants"  # eval only
    if row.top == "extra":                      return "extra"     # held-out hw
    # only `original` reaches here
    if row.hw in TEST_HW:                       return "test"
    if row.hw in VAL_HW:                        return "val"
    return "train"

m["label"]      = m.family.isin(RANSOM_FAMILIES).astype(int)
m["ransomware"] = m.family.where(m.family.isin(RANSOM_FAMILIES), "none")
m["mixed"]      = m.family.str.contains("_").astype(int)
m["split"]      = m.apply(split_of, axis=1)
m["sample_id"]  = m.rel_dir.str.replace("/", "__", regex=False)

print("=== rows per split × label ===")
print(m.groupby(["split", "label"]).size().unstack(fill_value=0))

print("\n=== rows per split × family ===")
print(m.groupby(["split", "family"]).size().unstack(fill_value=0).T)

m.to_csv("/kaggle/working/manifest.csv", index=False)
print("\n saved /kaggle/working/manifest.csv")

In [ ]:
# 3 executions per family from `original` on the smallest hw config
pool_orig = m[(m.top == "original") & (m.has_ata == 1)].copy()

# Choose the hw config with fewest executions (should be 120 per config)
hw_counts = pool_orig.groupby("hw").size().sort_values()
SMOKE_HW = hw_counts.index[0]
print(f"Smoke HW: {SMOKE_HW} ({hw_counts.iloc[0]} executions)")

sub = (pool_orig[pool_orig.hw == SMOKE_HW]
       .groupby("family", group_keys=False)
       .apply(lambda g: g.sample(min(3, len(g)), random_state=0))
       .reset_index(drop=True))

print(f"\nSmoke subset: {len(sub)} executions / {sub.total_bytes.sum()/1e9:.2f} GB")
print(sub.groupby("family").size().to_string())

sub.to_csv("/kaggle/working/subset_smoke.csv", index=False)

In [ ]:
import os, time, json
from pyspark.sql import functions as F

OUT_ROOT = "/kaggle/working/interim_smoke"
os.makedirs(OUT_ROOT, exist_ok=True)

def ingest_stream(stream, exec_df):
    out_path = f"{OUT_ROOT}/{stream}"
    if os.path.exists(f"{out_path}/_SUCCESS"):
        print(f"{stream}: already done")
        return None

    paths = [f"{RAW_ROOT}/{r}/{stream}.csv" for r in exec_df.rel_dir]

    df = (spark.read.option("header", False).schema(EVENT_DDL).csv(paths)
          .withColumn("_src", F.input_file_name()))

    # Strip everything up to and including ".../dataset/" → rel_dir
    df = (df
          .withColumn("rel_dir", F.regexp_extract(F.col("_src"), r"/dataset/(.+)/[^/]+$", 1))
          .drop("_src"))

    # Row validity: no nulls in required fields, size >= 0
    ok = (F.col("sec").isNotNull()
          & F.col("ns").isNotNull()
          & F.col("addr").isNotNull()
          & F.col("size").isNotNull()
          & (F.col("size") >= 0))

    # 1-second bucket index
    t   = F.col("sec") + F.col("ns") / 1e9
    win = F.floor(t / WINDOW_S).cast("long")

    v = lambda x: F.when(F.col("ok"), x)

    ev = df.withColumn("ok", ok).withColumn("win", win)

    aggs = [
        F.sum(F.col("ok").cast("int")).alias(f"{stream}_n"),
        F.count("*").alias(f"{stream}_rawcnt"),
        F.sum(v(F.col("size"))).alias(f"{stream}_size_sum"),
        (F.max(v(F.col("addr"))) - F.min(v(F.col("addr")))).alias(f"{stream}_addr_range"),
    ]
    if stream in HAS_ENTROPY:
        # entropy > 0 drops both -1.0 and -0.0 "unavailable" markers
        e = F.when(F.col("ok") & (F.col("entropy") > 0), F.col("entropy"))
        aggs += [
            F.mean(e).alias(f"{stream}_ent_mean"),
            F.max(e).alias(f"{stream}_ent_max"),
        ]

    (ev.groupBy("rel_dir", "win").agg(*aggs)
       .write.mode("overwrite").parquet(out_path))

    out = spark.read.parquet(out_path)
    s = out.agg(
        F.sum(f"{stream}_rawcnt").alias("raw"),
        F.sum(f"{stream}_n").alias("valid"),
        F.countDistinct("rel_dir").alias("executions"),
        F.count("*").alias("windows"),
    ).first()
    print(f"{stream:15s} exec={s['executions']:3d}  windows={s['windows']:6d}  "
          f"raw={s['raw']:>10,}  valid={s['valid']:>10,}  removed={s['raw']-s['valid']:>6,}")
    return {k: s[k] for k in ("raw", "valid", "executions", "windows")}

t0 = time.time()
report = {}
for s in STREAMS:
    report[s] = ingest_stream(s, sub)
print(f"\n ingestion done in {time.time()-t0:.0f}s")

with open("/kaggle/working/ingestion_report_smoke.json", "w") as fh:
    json.dump(report, fh, indent=2)

In [ ]:
BASE = "/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset"

print("=== Top-level breakdown ===")
for top in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, top)
    if not os.path.isdir(full): continue
    n_exec = 0
    n_bytes = 0
    for root, _, files in os.walk(full):
        csvs = [f for f in files if f.endswith(".csv")]
        if csvs:
            n_exec += 1
            n_bytes += sum(os.path.getsize(os.path.join(root, f)) for f in csvs)
    print(f"  {top:12s}  executions={n_exec:5d}   size={n_bytes/1e9:6.2f} GB")

print("\n=== CPU/RAM configs ===")
from collections import Counter
cfgs = Counter()
for rel in m.rel_dir:
    parts = rel.split("/")
    if len(parts) >= 3:
        cfgs[f"{parts[1]}/{parts[2]}"] += 1
for cfg, n in cfgs.most_common():
    print(f"  {cfg:35s} {n}")

In [ ]:
from functools import reduce
from pyspark.sql.window import Window

dfs = []
for s in STREAMS:
    p = f"{OUT_ROOT}/{s}"
    if os.path.exists(p):
        d = (spark.read.parquet(p)
             .filter(F.col("win").isNotNull())
             .drop(f"{s}_rawcnt"))
        dfs.append(d)

merged = reduce(lambda a, b: a.join(b, ["rel_dir", "win"], "full_outer"), dfs)
print(f"merged rows (before fill): {merged.count():,}")

# Build metadata to join in
meta_pd = sub[["rel_dir", "family", "top", "cpu", "ram", "hw", "label"]].copy()
meta    = spark.createDataFrame(meta_pd)

win = merged.join(F.broadcast(meta), "rel_dir", "inner")

# Fill numeric nulls with 0 = "no events in this stream during this window"
num_cols = [c for c in win.columns
            if c not in ("rel_dir", "win", "family", "top", "cpu", "ram", "hw", "label")]
win = win.fillna(0, subset=num_cols)

# step = window index from 0 for each execution
win = (win
       .withColumn("step",
                   F.col("win") - F.min("win").over(Window.partitionBy("rel_dir")))
       .drop("win"))

win.write.mode("overwrite").parquet(f"{OUT_ROOT}/windows")
n = spark.read.parquet(f"{OUT_ROOT}/windows").count()
print(f" windows written: {n:,}")

In [ ]:
d = spark.read.parquet(f"{OUT_ROOT}/windows")
print("columns:", d.columns, "\n")

# 1. Coverage
n_out = d.select("rel_dir").distinct().count()
print(f"executions: {n_out} / expected {len(sub)}")

# 2. Contiguity of step
per = (d.groupBy("rel_dir")
        .agg(F.count("*").alias("n"),
             F.min("step").alias("smin"),
             F.max("step").alias("smax")))
gaps = per.filter(F.col("smax") + 1 != F.col("n")).count()
print(f"executions with step gaps: {gaps}")

# 3. Negativity
feat_cols = [c for c in d.columns if c.endswith(("_n", "_size_sum", "_addr_range",
                                                 "_ent_mean", "_ent_max"))]
n_neg = d.filter(" OR ".join(f"`{c}` < 0" for c in feat_cols)).count()
print(f"rows with negative values: {n_neg}")

# 4. Windows per execution
print("\nwindows per execution:")
per.select(F.min("n").alias("min"),
           F.avg("n").alias("avg"),
           F.max("n").alias("max")).show()

# 5. Windows per family/label
print("windows per family:")
d.groupBy("family", "label").agg(
    F.countDistinct("rel_dir").alias("execs"),
    F.count("*").alias("windows"),
).orderBy("family").show()

# 6. Null-share per stream (must be 0)
print("null share per stream (should all be 0.0):")
d.select([F.avg(F.col(c).isNull().cast("int")).alias(c)
          for c in d.columns if c.endswith("_n")]).show(truncate=False)

assert n_out == len(sub), "coverage mismatch"
assert gaps == 0, f"{gaps} executions with step gaps"
assert n_neg == 0, f"{n_neg} negative values"
print("\n ALL VALIDATION RULES PASSED")

In [ ]:
d = spark.read.parquet(f"{OUT_ROOT}/windows")

# Row for one execution
one = d.filter(F.col("family") == "Conti").limit(10).toPandas()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
print(one.to_string())

In [ ]:
d = spark.read.parquet(f"{OUT_ROOT}/windows")

# Row for one execution
one = d.filter(F.col("family") == "Conti").limit(10).toPandas()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
print(one.to_string())

**feature-extraction/sequence-generation**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("cipherstop-feature-extraction")
    .config("spark.driver.memory", "20g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


In [ ]:
d = spark.read.parquet(
    "/kaggle/working/interim_smoke/windows"
)

print("Rows:", d.count())
print("Columns:", len(d.columns))

In [ ]:
d.printSchema()


In [ ]:
meta_cols = [
    "rel_dir",
    "family",
    "top",
    "cpu",
    "ram",
    "hw",
    "label",
    "step"
]

In [ ]:
feature_cols = [
    c for c in d.columns
    if c not in meta_cols
]

print("Feature count:", len(feature_cols))
print(feature_cols)

In [ ]:
ordered = d.orderBy(
    "rel_dir",
    "step"
)

In [ ]:
ordered = ordered.withColumn(
    "features",
    F.array(
        *[
            F.col(c).cast("double")
            for c in feature_cols
        ]
    )
)

In [ ]:
ordered.select(
    "rel_dir",
    "step",
    "features"
).show(5, truncate=False)

In [ ]:
seq_df = (
    ordered
    .groupBy(
        "rel_dir",
        "family",
        "label"
    )
    .agg(
        F.collect_list(
            F.struct(
                "step",
                "features"
            )
        ).alias("tmp")
    )
)

In [ ]:
seq_df = seq_df.withColumn(
    "tmp",
    F.array_sort("tmp")
)

In [ ]:
seq_df = seq_df.withColumn(
    "sequence",
    F.expr(
        """
        transform(
            tmp,
            x -> x.features
        )
        """
    )
)

In [ ]:
seq_df = seq_df.drop("tmp")

In [ ]:
seq_df.select(
    "family",
    "label",
    "sequence"
).show(3, truncate=False)

In [ ]:
seq_df = seq_df.withColumn(
    "seq_len",
    F.size("sequence")
)

seq_df.select(
    F.min("seq_len").alias("min"),
    F.avg("seq_len").alias("avg"),
    F.max("seq_len").alias("max")
).show()

In [ ]:
def cut_sequence(seq, ratio):

    if seq is None:
        return []

    n = len(seq)

    if n == 0:
        return []

    cut = max(
        1,
        int(n * ratio)
    )

    return seq[:cut]

In [ ]:
seq_type = ArrayType(
    ArrayType(
        DoubleType()
    )
)

In [ ]:
from pyspark.sql.functions import udf

cut10 = udf(
    lambda x: cut_sequence(x, 0.10),
    seq_type
)

cut25 = udf(
    lambda x: cut_sequence(x, 0.25),
    seq_type
)

cut50 = udf(
    lambda x: cut_sequence(x, 0.50),
    seq_type
)

cut75 = udf(
    lambda x: cut_sequence(x, 0.75),
    seq_type
)

In [ ]:
seq10 = seq_df.withColumn(
    "sequence",
    cut10("sequence")
)

In [ ]:
seq25 = seq_df.withColumn(
    "sequence",
    cut25("sequence")
)

In [ ]:
seq50 = seq_df.withColumn(
    "sequence",
    cut50("sequence")
)

In [ ]:
seq75 = seq_df.withColumn(
    "sequence",
    cut75("sequence")
)

In [ ]:
for name, dataset in [
    ("10%", seq10),
    ("25%", seq25),
    ("50%", seq50),
    ("75%", seq75)
]:

    print("\n", name)

    dataset.select(
        F.size("sequence").alias(
            "length"
        )
    ).show(5)

In [ ]:
seq10.write.mode("overwrite").parquet(
    "/kaggle/working/processed/10pct"
)

seq25.write.mode("overwrite").parquet(
    "/kaggle/working/processed/25pct"
)

seq50.write.mode("overwrite").parquet(
    "/kaggle/working/processed/50pct"
)

seq75.write.mode("overwrite").parquet(
    "/kaggle/working/processed/75pct"
)

In [ ]:
print("Original :", seq_df.count())
print("10%      :", seq10.count())
print("25%      :", seq25.count())
print("50%      :", seq50.count())
print("75%      :", seq75.count())

In [ ]:
import os

for root, dirs, files in os.walk(
    "/kaggle/working/processed"
):
    print(root)